# SWE-E2E v2 dump browser

Browses everything dumped by the vLLM / MINF SingleController runs under
`/scratch/fsw/portfolios/nemotron/projects/nemotron_sw_post/users/rkirby/runs/<EXP>/`:

| what | where | granularity |
|---|---|---|
| rollout dumps | `dumps/rollouts/target_step_NNNNN.jsonl` (~10 GB/step) | one JSON per rollout: ids, reward, per-message token counts, the full Gym result (every turn's text, tool calls, per-turn token usage, timings, error kind) |
| token-level dumps | `dumps/token_level/step_NNNNN_chunk_NNN.pt` | per sequence: input_ids, token_mask, generation logprobs, trainer logprobs, advantages, rewards, sample masks |
| precomputed summaries | `swe_dump/analysis/rollouts/*.summary.jsonl`, `analysis/tokens/*.rows.csv`, `*.hist.json` | one row per rollout / sequence (fast to load) |
| W&B scalars (cached) | `analysis/wandb_*.json` | per train step |
| Gym per-instance results (runs without dumps, e.g. the main MINF chain) | `swe_minf/nemo_rl/3rdparty/Gym-workspace/Gym/results/swebench_results_<session>/<instance>_<ts>_<uuid>/nemo_gym_metrics.json` | one dir per rollout |

Naming: rollout files are keyed by **target_step k**, which feeds **train step k+1**; token-level files and W&B use the train step.
All helpers below take the train step.

Reports already written: `analysis/aggregate_report.md`, `analysis/deep_report.md`, `analysis/ab/` (parser A/B), `analysis/early_steps/report.md`.
Regenerate summaries for new steps/runs (CPU job templates in `analysis/logs/*.sbatch`):

```
python3 tools/rollout_summary.py analysis/rollouts <run_dir>/dumps/rollouts/target_step_*.jsonl
python3 tools/token_level_summary.py analysis/tokens <run_dir>/dumps/token_level/step_*_chunk_*.pt
```


In [ ]:
import os, sys, json, glob, re, statistics as st
from collections import Counter, defaultdict
import numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80); pd.set_option("display.max_colwidth", 140)
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception:
    HAVE_MPL = False
    print("matplotlib not available: tables only")

W = "/scratch/fsw/portfolios/nemotron/projects/nemotron_sw_post/users/rkirby/workspaces/swe_dump"
RUNS_ROOT = "/scratch/fsw/portfolios/nemotron/projects/nemotron_sw_post/users/rkirby/runs"
ANALYSIS = f"{W}/analysis"
GYM_MAIN = "/scratch/fsw/portfolios/nemotron/projects/nemotron_sw_post/users/rkirby/workspaces/swe_minf/nemo_rl/3rdparty/Gym-workspace/Gym/results"
sys.path.insert(0, f"{W}/tools")
from pt_numpy import load as load_pt   # torch-free loader for the .pt dumps

RUNS = {
    "runA_vllm": "nano35-swe-v2-stream128-inorder1-cmh-64n-vllm_dump-20260918",
    "runB_minf": "nano35-swe-v2-stream128-inorder1-cmh-64n-minf_dump-from10-20260918",
    "chainC_minf_nopg_noprefix": "nano35-swe-v2-stream128-inorder1-cmh-64n-minf_dump-nopg-noprefix-20260919",
    "chainD_vllm_from10_nopg": "nano35-swe-v2-stream128-inorder1-cmh-64n-vllm_dump-from10-nopg-20260919",
    "main_minf": "nano35-swe-v2-stream128-inorder1-cmh-64n-minf",
}
EXP2RUN = {v: k for k, v in RUNS.items()}

def run_dir(run):
    return f"{RUNS_ROOT}/{RUNS.get(run, run)}"

def rollout_files(run):
    return sorted(glob.glob(f"{run_dir(run)}/dumps/rollouts/target_step_*.jsonl"))

def token_files(run, train_step=None):
    pat = f"step_{train_step:05d}_chunk_*.pt" if train_step is not None else "step_*_chunk_*.pt"
    return sorted(glob.glob(f"{run_dir(run)}/dumps/token_level/{pat}"))

def step_of(path):
    m = re.search(r"(?:target_step|step)_(\d+)", os.path.basename(path))
    return int(m.group(1)) if m else None


## Inventory

In [ ]:
def inventory():
    rows = []
    for run in RUNS:
        d = run_dir(run); rf = rollout_files(run); tf = token_files(run)
        ck = sorted(int(p.rsplit("_", 1)[1]) for p in glob.glob(f"{d}/checkpoints/step_*") if p.rsplit("_", 1)[1].isdigit())
        rows.append(dict(run=run,
                         rollout_train_steps=f"{step_of(rf[0]) + 1}-{step_of(rf[-1]) + 1}" if rf else "-", rollout_files=len(rf),
                         rollout_GB=round(sum(os.path.getsize(p) for p in rf) / 1e9, 1),
                         token_train_steps=f"{step_of(tf[0])}-{step_of(tf[-1])}" if tf else "-", token_chunks=len(tf),
                         token_GB=round(sum(os.path.getsize(p) for p in tf) / 1e9, 1),
                         checkpoints=ck[-6:], gym_sessions=len(glob.glob(f"{d}/gym_results/*"))))
    return pd.DataFrame(rows)

inventory()


## Precomputed per-rollout and per-sequence summaries

Fast path: one row per rollout (`R`) and one row per trained sequence (`T`). Join on `sample_id`.

In [ ]:
def load_rollout_summaries(pattern="*"):
    recs = []
    for f in sorted(glob.glob(f"{ANALYSIS}/rollouts/{pattern}.summary.jsonl")):
        with open(f) as fh:
            recs.extend(json.loads(line) for line in fh if line.strip())
    R = pd.DataFrame(recs)
    R = R[R["sample_id"].notna()].copy()
    R["train_step"] = R["target_step"].astype(int) + 1
    R["short_run"] = R["run"].map(EXP2RUN).fillna(R["run"])
    R["engine"] = np.where(R["run"].str.contains("vllm"), "vllm", "minf")
    R["truncated"] = R["truncated"].fillna(False).astype(bool)
    return R

def load_token_rows(pattern="*"):
    files = sorted(glob.glob(f"{ANALYSIS}/tokens/{pattern}.rows.csv"))
    T = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    T["train_step"] = T["file"].str.extract(r"step_(\d+)").astype(int)
    T["short_run"] = T["run"].map(EXP2RUN).fillna(T["run"])
    T["engine"] = np.where(T["run"].str.contains("vllm"), "vllm", "minf")
    return T

R = load_rollout_summaries()
T = load_token_rows()
print("rollouts", R.shape, " sequences", T.shape)
R.groupby("short_run")["train_step"].agg(["min", "max", "count"])


In [ ]:
R[R['n_gen_without_think_close'] > 0]

### Per-step aggregates

In [ ]:
def per_step(R):
    g = R.groupby(["short_run", "train_step"])
    out = pd.DataFrame({
        "n": g.size(), "reward": g["reward"].mean(),
        "gen_len_mean": g["gen_len"].mean(), "gen_len_p50": g["gen_len"].median(), "gen_len_p95": g["gen_len"].quantile(0.95),
        "trunc_rate": g["truncated"].mean(), "turns_mean": g["turns"].mean(), "max_turn_tokens_mean": g["max_asst_tokens"].mean(),
        "ctx_window_rate": g["agent_error_kind"].apply(lambda s: (s == "context_window").mean()),
        "max_iter_rate": g["agent_error_kind"].apply(lambda s: (s == "max_iteration").mean()),
    })
    return out.round(3)

PS = per_step(R)
PS


In [ ]:
if HAVE_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for run, d in PS.reset_index().groupby("short_run"):
        for ax, col in zip(axes, ["gen_len_mean", "reward", "trunc_rate"]):
            ax.plot(d["train_step"], d[col], marker="o", label=run); ax.set_title(col); ax.set_xlabel("train step")
    axes[0].legend(fontsize=7); plt.tight_layout()


### Group view for one step

Pass rate, success/failure lengths and advantages per prompt group (advantages come from the token rows).

In [ ]:
def groups(run, train_step):
    exp = RUNS.get(run, run)
    r = R[(R.run == exp) & (R.train_step == train_step)]
    t = T[(T.run == exp) & (T.train_step == train_step)][["sample_id", "adv", "n_gen_tokens", "sample_mask_after", "lp_err_mean", "seq_mult_prob_error"]]
    m = r.merge(t, on="sample_id", how="left")
    rows = []
    for (gid, inst), d in m.groupby(["group_id", "instance_id"]):
        rows.append(dict(group_id=gid, instance_id=inst, n=len(d), pass_rate=d.reward.mean(),
                         L_succ=d.loc[d.reward > 0, "gen_len"].mean(), L_fail=d.loc[d.reward <= 0, "gen_len"].mean(),
                         adv_min=d.adv.min(), adv_max=d.adv.max(), turns=d.turns.mean(), trunc=int(d.truncated.sum()),
                         masked=int((d.sample_mask_after == 0).sum()), errors=dict(Counter(d.agent_error_kind.dropna()))))
    return pd.DataFrame(rows).sort_values("pass_rate")

G = groups("runB_minf", 13)
G.head(40)


## Raw rollouts (streamed from the jsonl dumps)

`iter_rollouts` streams a step file (10 GB), `get_rollout` stops at the first match, `show_rollout` prints the turn structure from the Gym result: reasoning text, assistant message, tool call (with the raw `generation_str`), tool output.

In [ ]:
def iter_rollouts(run, train_step):
    path = f"{run_dir(run)}/dumps/rollouts/target_step_{train_step - 1:05d}.jsonl"
    with open(path, "rb") as fh:
        for line in fh:
            if line.strip():
                yield json.loads(line)

def get_rollout(run, train_step, sample_id=None, instance_id=None, completion_index=None):
    for r in iter_rollouts(run, train_step):
        if sample_id is not None and r["sample_id"] == sample_id:
            return r
        if instance_id is not None and r["prompt_metadata"].get("instance_id") == instance_id and (completion_index is None or r["completion_index"] == completion_index):
            return r
    return None

def item_text(o):
    c = o.get("content")
    if isinstance(c, list):
        return "".join(str(p.get("text", "")) for p in c if isinstance(p, dict))
    if c is None and o.get("summary"):
        return "".join(str(p.get("text", "")) for p in o["summary"] if isinstance(p, dict))
    return str(c if c is not None else o.get("output", ""))

def output_items(r):
    return ((r.get("full_result") or {}).get("response") or {}).get("output") or []

def turn_generations(r):
    return [o.get("generation_str") or "" for o in output_items(r) if o.get("type") == "function_call"]

def show_rollout(r, max_items=40, max_chars=600):
    fr = r.get("full_result") or {}
    print(f"sample {r['sample_id']}  instance {r['prompt_metadata'].get('instance_id')}  train_step {r['target_step'] + 1}  reward {r['reward']}  "
          f"gen_len {r['generation_length']}  total {r['total_tokens']}  turns {r['num_assistant_turns']}  truncated {r['truncated']}  "
          f"error {fr.get('agent_error_kind')}  resolved {fr.get('resolved')}  weight_version {r['start_weight_version']}->{r['end_weight_version']}")
    print("messages:", dict(Counter(m["role"] for m in r["messages"])),
          " tokens by role:", {k: sum(m["n_tokens"] for m in r["messages"] if m["role"] == k) for k in ("system", "user", "assistant", "tool")},
          " flagged:", sum(bool(m.get("is_invalid_tool_call")) for m in r["messages"]), "invalid tool calls,",
          sum(bool(m.get("has_malformed_thinking")) for m in r["messages"]), "malformed thinking")
    out = output_items(r)
    for i, o in enumerate(out[:max_items]):
        t = o.get("type")
        if t == "function_call":
            g = o.get("generation_str") or ""
            print(f"\n[{i}] function_call {o.get('name')}  status={o.get('status')}  generation_chars={len(g)}  think_closed={'</think>' in g}")
            print("    args:", str(o.get("arguments"))[:max_chars])
        else:
            print(f"\n[{i}] {t}: {item_text(o)[:max_chars]!r}")
    if len(out) > max_items:
        print(f"\n... {len(out) - max_items} more items")

r0 = next(iter_rollouts("runB_minf", 13))
show_rollout(r0, max_items=10)


In [ ]:
def long_turns(run, train_step, min_chars=20000, limit=25):
    rows = []
    for r in iter_rollouts(run, train_step):
        for k, g in enumerate(turn_generations(r)):
            if len(g) >= min_chars:
                rows.append(dict(sample_id=r["sample_id"], instance=r["prompt_metadata"].get("instance_id"), turn=k, chars=len(g),
                                 reward=r["reward"], think_closed="</think>" in g, has_tool_call="<tool_call>" in g,
                                 error=(r.get("full_result") or {}).get("agent_error_kind")))
    return pd.DataFrame(rows).sort_values("chars", ascending=False).head(limit)

# scans a whole step file (~10 GB, a few minutes); uncomment to run
# long_turns("runB_minf", 13)


## Token-level dumps

Each chunk holds ~170 sequences: flat `input_ids` / `token_mask` with per-sequence `input_lengths`, and `generation_logprobs` / `prev_logprobs` / `advantages` with `input_lengths - 1` entries per sequence (entry j scores token j+1). `token_mask` marks generated tokens.

In [ ]:
def load_chunk(path):
    d = load_pt(path)
    lengths = np.asarray(d["input_lengths"]).astype(np.int64)
    d["_ids_off"] = np.concatenate([[0], np.cumsum(lengths)])
    d["_lp_off"] = np.concatenate([[0], np.cumsum(lengths - 1)])
    return d

def sequence(d, i):
    a, b = d["_ids_off"][i], d["_ids_off"][i + 1]
    c, e = d["_lp_off"][i], d["_lp_off"][i + 1]
    tags = d.get("tags") or [{}] * len(d["sample_ids"])
    return dict(sample_id=d["sample_ids"][i], tags=tags[i], reward=float(d["rewards"][i]),
                ids=np.asarray(d["input_ids"])[a:b],
                gen=np.asarray(d["token_mask"])[a + 1:b].astype(bool),
                gen_lp=np.asarray(d["generation_logprobs"])[c:e].astype(np.float32),
                prev_lp=np.asarray(d["prev_logprobs"])[c:e].astype(np.float32),
                adv=np.asarray(d["advantages"])[c:e].astype(np.float32),
                sample_mask_before=float(d["sample_mask_before"][i]), sample_mask_after=float(d["sample_mask_after"][i]),
                seq_mult_prob_error=float(d["seq_mult_prob_error"][i]))

def find_sample(run, train_step, sample_id):
    for p in token_files(run, train_step):
        d = load_chunk(p)
        ids = list(d["sample_ids"])
        if sample_id in ids:
            return sequence(d, ids.index(sample_id))
    return None

try:
    from tokenizers import Tokenizer
    TOK = Tokenizer.from_file(f"{RUNS_ROOT}/nano35-swe-v2-stream128-inorder1-cmh-64n-minf/checkpoints/step_55/hf/tokenizer.json")
    def decode(ids):
        return TOK.decode([int(x) for x in ids], skip_special_tokens=False)
except Exception as exc:
    TOK = None
    print("tokenizer not importable here (needs the `tokenizers` package):", exc)
    def decode(ids):
        return "<tokenizer unavailable>"

files = token_files("runB_minf", 13)
print(len(files), "chunks;", [os.path.basename(f) for f in files])


In [ ]:
import itertools
for r0 in itertools.islice(iter_rollouts("runB_minf", 13), 5):
    s = find_sample("runB_minf", 13, r0["sample_id"])
    if s:
        g = s["gen"]
        print(s["sample_id"], "tags", s["tags"])
        print(f"tokens {len(s['ids'])}  generated {int(g.sum())}  reward {s['reward']}  adv {s['adv'][g][0]:+.3f}  mask_after {s['sample_mask_after']}")
        print(f"mean gen_lp {s['gen_lp'][g].mean():+.4f}  mean prev_lp {s['prev_lp'][g].mean():+.4f}  mean(gen-prev) {(s['gen_lp'] - s['prev_lp'])[g].mean():+.5f}  seq_mult_prob_error {s['seq_mult_prob_error']:.4f}")
        err = np.abs(s["gen_lp"] - s["prev_lp"]) * g
        for j in np.argsort(-err)[:8]:
            print(f"  pos {j:6d}  token {decode(s['ids'][j + 1:j + 2])!r:20}  gen_lp {s['gen_lp'][j]:+.3f}  prev_lp {s['prev_lp'][j]:+.3f}")
        if HAVE_MPL:
            fig, ax = plt.subplots(figsize=(14, 3))
            pos = np.arange(len(s["gen_lp"]))[g]
            ax.plot(pos, (s["gen_lp"] - s["prev_lp"])[g], lw=0.4); ax.set_title("generation logprob - trainer logprob along the sequence (generated tokens)"); ax.set_xlabel("position")


In [ ]:
import itertools
step = 1
for r0 in itertools.islice(iter_rollouts("runA_vllm", step), 5):
    s = find_sample("runA_vllm", step, r0["sample_id"])
    if s:
        g = s["gen"]
        print(s["sample_id"], "tags", s["tags"])
        print(f"tokens {len(s['ids'])}  generated {int(g.sum())}  reward {s['reward']}  adv {s['adv'][g][0]:+.3f}  mask_after {s['sample_mask_after']}")
        print(f"mean gen_lp {s['gen_lp'][g].mean():+.4f}  mean prev_lp {s['prev_lp'][g].mean():+.4f}  mean(gen-prev) {(s['gen_lp'] - s['prev_lp'])[g].mean():+.5f}  seq_mult_prob_error {s['seq_mult_prob_error']:.4f}")
        err = np.abs(s["gen_lp"] - s["prev_lp"]) * g
        for j in np.argsort(-err)[:8]:
            print(f"  pos {j:6d}  token {decode(s['ids'][j + 1:j + 2])!r:20}  gen_lp {s['gen_lp'][j]:+.3f}  prev_lp {s['prev_lp'][j]:+.3f}")
        if HAVE_MPL:
            fig, ax = plt.subplots(figsize=(14, 3))
            pos = np.arange(len(s["gen_lp"]))[g]
            ax.plot(pos, (s["gen_lp"] - s["prev_lp"])[g], lw=0.4); ax.set_title("generation logprob - trainer logprob along the sequence (generated tokens)"); ax.set_xlabel("position")


### Generation-vs-trainer logprob error histograms (from `*.hist.json`)

In [ ]:
H = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{ANALYSIS}/tokens/*.hist.json"))])
H["train_step"] = H["file"].str.extract(r"step_(\d+)").astype(int)
H["short_run"] = H["run"].map(EXP2RUN).fillna(H["run"])
bins = H["lp_err_bins"].iloc[0]
agg = {run: np.sum(np.vstack(d["lp_err_hist"].values), axis=0) for run, d in H.groupby("short_run")}
pd.DataFrame({run: v / v.sum() for run, v in agg.items()}, index=[f"[{bins[i]}, {bins[i + 1]})" for i in range(len(bins) - 1)]).round(6)


## W&B scalars (cached pulls)

`wandb_step_metrics_full.json` holds main_minf / runA_vllm / runB_minf; `wandb_engine_compare.json` holds the vLLM baseline (`vllm_base`), the main MINF chain and a second vLLM run. Refresh with the GraphQL pull used before (needs `WANDB_API_KEY` in the environment; never print it).

In [ ]:
def load_wandb():
    frames = []
    for f in sorted(glob.glob(f"{ANALYSIS}/wandb_*.json")):
        d = json.load(open(f))
        for run, steps in d.items():
            if not isinstance(steps, dict):
                continue
            df = pd.DataFrame.from_dict(steps, orient="index")
            df.index = df.index.astype(int); df.index.name = "train_step"
            df["run"] = run; df["source"] = os.path.basename(f)
            frames.append(df.reset_index())
    return pd.concat(frames, ignore_index=True)

WB = load_wandb()
KEYS = ["train/reward", "train/rollout_length/swe_agents_train/mean", "train/rollout_length/swe_agents_train/truncation_rate",
        "train/advantages/mean", "train/gen_kl_error", "train/total_num_tokens", "train/num_masked_seqs_by_logprob_error"]
WB[WB.source == "wandb_engine_compare.json"].pivot_table(index="train_step", columns="run", values="train/advantages/mean").head(30)


In [ ]:
if HAVE_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    for ax, key in zip(axes.flat, ["train/rollout_length/swe_agents_train/mean", "train/reward", "train/advantages/mean", "train/gen_kl_error"]):
        for (src, run), d in WB.groupby(["source", "run"]):
            if key in d and d[key].notna().any():
                ax.plot(d["train_step"], d[key], marker=".", lw=0.8, label=f"{run} ({src.split('_')[1]})")
        ax.set_title(key); ax.set_xlabel("train step")
    axes[0, 0].legend(fontsize=6); plt.tight_layout()


## Gym per-instance results (for runs without dumps, e.g. the main MINF chain)

One directory per rollout; `nemo_gym_metrics.json` has `resolved`, per-turn token usage (`completion_tokens` equals the trained generation length), timings and `agent_error_kind`. Sessions map to Slurm segments by mtime.

In [ ]:
def gym_sessions(root=GYM_MAIN):
    s = sorted(glob.glob(f"{root}/swebench_results_*"), key=os.path.getmtime)
    return pd.DataFrame([(os.path.basename(p), len(os.listdir(p)), pd.Timestamp(os.path.getmtime(p), unit="s", tz="America/Los_Angeles")) for p in s],
                        columns=["session", "instances", "last_write"])

def gym_results_df(session_dir, limit=None):
    rows = []
    for d in os.scandir(session_dir):
        p = os.path.join(d.path, "nemo_gym_metrics.json")
        if not os.path.exists(p):
            continue
        j = json.load(open(p))
        m = re.match(r"^(.*)_(\d{13})_([0-9a-f]{8})$", d.name)
        u = ((j.get("per_turn_metrics") or {}).get("accumulated_token_usage") or {})
        rows.append(dict(instance=m.group(1) if m else d.name, resolved=j.get("resolved"), completion_tokens=u.get("completion_tokens"),
                         prompt_tokens=u.get("prompt_tokens"), turns=len((j.get("per_turn_metrics") or {}).get("token_usages") or []),
                         error=j.get("agent_error_kind"), start=j.get("generation_start_timestamp"), run_time=j.get("openhands_run_time"), dir=d.name))
        if limit and len(rows) >= limit:
            break
    return pd.DataFrame(rows)

gym_sessions()


In [ ]:
# main MINF chain, segment 1 (train steps 1-8, 2026-09-16 07:53-11:53 PDT) -- ~1 min to scan
# GM = gym_results_df(f"{GYM_MAIN}/swebench_results_1789570827574_f7cb6b6f")
# GM.groupby("resolved")["completion_tokens"].describe()


## Early-step engine comparison (steps 1-8, same prompts, same weights)

Output of `tools/early_step_compare.py` / `tools/early_step_mixed.py`: per prompt, run A (vLLM, `A_*`) against the main MINF chain (`M_*`).

In [ ]:
EP = pd.read_csv(f"{ANALYSIS}/early_steps/per_prompt.csv")
EP["dL_rel"] = (EP.M_meanL - EP.A_meanL) / EP.A_meanL
EP["d_pass"] = EP.M_k / EP.M_n - EP.A_k / EP.A_n
EP.groupby("step")[["dL_rel", "d_pass"]].agg(["mean", "median", "count"]).round(3)


## Other tools in `swe_dump/tools/`

- `aggregate_dumps.py` (engine-level aggregate report), `token_deep.py` + `aggregate_deep.py` (per-token category / position / turn stats), `parser_ab.py` (re-parse every turn with both tool parsers)
- `runaway_detail.py`, `inspect_row.py`, `message_turns.py`, `user_message_context.py`, `error_strings.py` (single-rollout inspection helpers)
- `early_step_compare.py`, `early_step_mixed.py` (this notebook's last section)
